In [ ]:
import json
import torch
import torch.distributed.fsdp
class FakeFSDPModule:
    pass
torch.distributed.fsdp.FSDPModule = FakeFSDPModule
from unsloth import FastLanguageModel
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
MODEL_PATH = "biling_model"
DEV_PATH = "merge_selected.jsonl"

OUTPUT_TSV = "biling_inf/predictions_rel.tsv"
OUTPUT_CSV = "biling_inf/predictions_analysis.csv"

DEVICE = "cuda"
MAX_SEQ_LENGTH = 1024
MAX_PROMPT_TOKENS = 900

LABELS = [
    "ABBREVIATION", "ALTERNATIVE_NAME", "SUBCLASS_OF", "PART_OF",
    "TREATED_USING", "ORIGINS_FROM", "TO_DETECT_OR_STUDY", "AFFECTS",
    "HAS_CAUSE", "APPLIED_TO", "USED_IN", "ASSOCIATED_WITH",
    "PHYSIOLOGY_OF", "FINDING_OF",
    "no_relation"
]

relation_list_str = ", ".join(LABELS)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
model.eval()

In [ ]:
dev_data = []
with open(DEV_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            dev_data.append(json.loads(line))

print(f"Loaded {len(dev_data)} examples from {DEV_PATH}")

In [ ]:
prompt_template = """You are an expert in biomedical information extraction.
Analyze the text and determine the relation between the two specified entities.
You must choose ONLY ONE relation from the following list:
[{relation_list}]

Text: {text}
Entity 1 (Head): {head_name} (Type: {head_type})
Entity 2 (Tail): {tail_name} (Type: {tail_type})
Relation:"""

def get_doc_id(item):
    return item.get("doc_id", "unknown")

def get_head_name(item):
    return item["h"]["name"]

def get_tail_name(item):
    return item["t"]["name"]

def get_head_type(item):
    return item.get("head_type", item.get("h", {}).get("type", "UNK"))

def get_tail_type(item):
    return item.get("tail_type", item.get("t", {}).get("type", "UNK"))

def get_head_span(item):
    return item.get("head_span", "")

def get_tail_span(item):
    return item.get("tail_span", "")

def get_gold_relation(item):
    return item.get("relation", None)

def build_prompt(item):
    return prompt_template.format(
        relation_list=relation_list_str,
        text=item["text"],
        head_name=get_head_name(item),
        head_type=get_head_type(item),
        tail_name=get_tail_name(item),
        tail_type=get_tail_type(item),
    )

In [ ]:
label_token_ids = {}
for label in LABELS:
    ids = tokenizer(" " + label, add_special_tokens=False)["input_ids"]
    label_token_ids[label] = ids

In [ ]:
@torch.no_grad()
def predict_label_batched(prompt: str, labels):
    """
    Для одного prompt считает все labels одним батчем.
    Без ручного past_key_values.
    """

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_PROMPT_TOKENS,
    )["input_ids"]

    prompt_len = len(prompt_ids)

    batch_input_ids = []
    batch_attention_mask = []
    meta = []

    max_len = 0

    for label in labels:
        suffix_ids = label_token_ids[label]

        full_ids = prompt_ids + suffix_ids

        full_ids = full_ids[:MAX_SEQ_LENGTH]

        actual_suffix_len = max(0, len(full_ids) - prompt_len)

        batch_input_ids.append(full_ids)
        batch_attention_mask.append([1] * len(full_ids))
        meta.append({
            "label": label,
            "prompt_len": prompt_len,
            "suffix_len": actual_suffix_len,
            "suffix_ids": suffix_ids[:actual_suffix_len],
        })

        if len(full_ids) > max_len:
            max_len = len(full_ids)

    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id

    padded_input_ids = []
    padded_attention_mask = []

    for ids, mask in zip(batch_input_ids, batch_attention_mask):
        pad_len = max_len - len(ids)
        padded_input_ids.append(ids + [pad_id] * pad_len)
        padded_attention_mask.append(mask + [0] * pad_len)

    input_ids = torch.tensor(padded_input_ids, dtype=torch.long, device=DEVICE)
    attention_mask = torch.tensor(padded_attention_mask, dtype=torch.long, device=DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        return_dict=True,
    )

    logits = outputs.logits
    log_probs = torch.log_softmax(logits, dim=-1)

    scores = []

    for i, m in enumerate(meta):
        label = m["label"]
        suffix_ids = m["suffix_ids"]
        suffix_len = m["suffix_len"]
        p_len = m["prompt_len"]

        if suffix_len == 0:
            scores.append({
                "label": label,
                "total_logprob": -1e9,
                "avg_logprob": -1e9,
                "num_tokens": 0,
            })
            continue

        total_logprob = 0.0

        for j in range(suffix_len):
            token_pos = p_len + j
            token_id = suffix_ids[j]

            if token_pos - 1 >= logits.shape[1]:
                total_logprob = -1e9
                break

            token_logprob = log_probs[i, token_pos - 1, token_id].item()
            total_logprob += token_logprob

        avg_logprob = total_logprob / max(suffix_len, 1)

        scores.append({
            "label": label,
            "total_logprob": total_logprob,
            "avg_logprob": avg_logprob,
            "num_tokens": suffix_len,
        })

    scores = sorted(scores, key=lambda x: x["avg_logprob"], reverse=True)
    best = scores[0]
    margin = scores[0]["avg_logprob"] - scores[1]["avg_logprob"] if len(scores) > 1 else None

    return best["label"], scores, margin

In [ ]:
analysis_rows = []
correct_predictions = 0

with open(OUTPUT_TSV, "w", encoding="utf-8") as tsv_file:
    tsv_file.write("document_id\trelation\thead_text\thead_span\thead_type\ttail_text\ttail_span\ttail_type\n")

    for item in tqdm(dev_data, desc="Batched logprob inference"):
        prompt = build_prompt(item)
        pred_label, scores, margin = predict_label_batched(prompt, LABELS)

        gold_label = get_gold_relation(item)
        is_correct = (pred_label == gold_label)
        if is_correct:
            correct_predictions += 1

        tsv_file.write(
            f"{get_doc_id(item)}\t"
            f"{pred_label}\t"
            f"{get_head_name(item)}\t"
            f"{get_head_span(item)}\t"
            f"{get_head_type(item)}\t"
            f"{get_tail_name(item)}\t"
            f"{get_tail_span(item)}\t"
            f"{get_tail_type(item)}\n"
        )

        row = {
            "document_id": get_doc_id(item),
            "gold_label": gold_label,
            "pred_label": pred_label,
            "is_correct": is_correct,

            "text": item["text"],
            "head_text": get_head_name(item),
            "head_span": get_head_span(item),
            "head_type": get_head_type(item),
            "tail_text": get_tail_name(item),
            "tail_span": get_tail_span(item),
            "tail_type": get_tail_type(item),

            "prompt": prompt,
            "confidence_margin": margin,
            "raw_input_json": json.dumps(item, ensure_ascii=False),
        }

        for s in scores:
            row[f"score_total__{s['label']}"] = s["total_logprob"]
            row[f"score_avg__{s['label']}"] = s["avg_logprob"]
            row[f"num_tokens__{s['label']}"] = s["num_tokens"]

        analysis_rows.append(row)

In [ ]:
df = pd.DataFrame(analysis_rows)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")